# 09 - Plugin System and Hook Lifecycle

> **When to use**: When you need to extend sqlseed's functionality (e.g., auto-add timestamps, data masking, custom generators).
>
> **Core concept**: pluggy plugin framework, 11 Hooks covering the full lifecycle from registration to writing.

## Applicable Scenarios

- Auto-add fields per row (e.g., `created_at`) → `transform_row` Hook
- Batch data transformation (e.g., uppercase all) → `transform_batch` Hook
- Custom data generators → `register_providers` Hook
- Run logic before/after writing → `before_insert` / `after_insert` Hook

## What You Will Learn

- 11 Hook lifecycle
- Custom Provider development
- PluginMediator bridging mechanism
- entry-point packaging flow

See architecture.zh-CN.md §8

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| **→ 09** | **Plugin System and Hook Lifecycle** | **Plugins** | **01** |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [1]:
from sqlseed.generators._dispatch import GeneratorDispatchMixin
from sqlseed.generators._protocol import DataProvider
from sqlseed.plugins.hookspecs import hookimpl

import sqlite3
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Plugin Hook Specs | `src/sqlseed/plugins/hookspecs.py` | `SqlseedHookSpec` |

> Corresponding architecture diagram: [§8 Plugin Hook Lifecycle](../docs/architecture.zh-CN.md#8-插件-hook-生命周期)

## 1. See It in Action — Plugin Auto-Adds Timestamps

Register a plugin that auto-adds a `created_at` timestamp before each row is written — **zero intrusion**:

```python
class TimestampPlugin:
    @hookimpl
    def sqlseed_transform_row(self, table_name, row):
        row['created_at'] = datetime.now().isoformat()
        return row
```

Below we demo plugin development and the 11 Hook lifecycle.

## 2. Custom Provider Development

A custom Provider must implement the `DataProvider` Protocol and inherit from `GeneratorDispatchMixin`:

In [2]:
import secrets
import typing


class ChineseNameProvider(GeneratorDispatchMixin, DataProvider):
    name = "chinese_name"
    _GENERATOR_MAP: typing.ClassVar[dict] = {}

    def __init__(self):
        self._locale = "zh"
        self._seed = None
        self._last_names = ["张", "王", "李", "赵", "刘", "陈", "杨", "黄", "周", "吴"]
        self._first_names = ["伟", "芳", "秀英", "敏", "静", "丽", "强", "磊", "洋", "勇"]

    def set_locale(self, locale: str) -> None:
        self._locale = locale

    def set_seed(self, seed: int | None) -> None:
        self._seed = seed

    def generate(self, type_name: str, **params: object) -> object:
        last = secrets.choice(self._last_names)
        first = secrets.choice(self._first_names)
        return f"{last}{first}"


provider = ChineseNameProvider()
print(f"Provider: {provider.name}")
print(f"Sample: {provider.generate('name')}")
print(f"Sample: {provider.generate('name')}")
print(f"Sample: {provider.generate('name')}")

Provider: chinese_name
Sample: 赵磊
Sample: 黄伟
Sample: 刘磊


## 3. 11 Hook Lifecycle

sqlseed defines 11 Hooks, grouped by execution phase:

| Phase | Hook | Description |
|---|---|---|
| Register | sqlseed_register_providers | Register custom Provider |
| Register | sqlseed_register_column_mappers | Register custom column mapping rules |
| AI Analysis | sqlseed_ai_analyze_table | AI analyzes table structure (firstresult) |
| AI Analysis | sqlseed_pre_generate_templates | Pre-generate template values (firstresult) |
| Generate | sqlseed_before_generate | Pre-generation callback |
| Generate | sqlseed_after_generate | Post-generation callback |
| Generate | sqlseed_transform_row | Per-row transform (hot path) |
| Generate | sqlseed_transform_batch | Batch transform (chained) |
| Write | sqlseed_before_insert | Pre-insert callback |
| Write | sqlseed_after_insert | Post-insert callback |
| Shared Pool | sqlseed_shared_pool_loaded | Shared pool load complete |

In [3]:
hookspec_names = [
    ("sqlseed_register_providers", "Register", "Register custom Provider"),
    ("sqlseed_register_column_mappers", "Register", "Register custom column mapping rules"),
    ("sqlseed_ai_analyze_table", "AI Analysis", "AI analyzes table structure (firstresult)"),
    ("sqlseed_pre_generate_templates", "AI Analysis", "Pre-generate template values (firstresult)"),
    ("sqlseed_before_generate", "Generate", "Pre-generation callback"),
    ("sqlseed_after_generate", "Generate", "Post-generation callback"),
    ("sqlseed_transform_row", "Generate", "Per-row transform (hot path)"),
    ("sqlseed_transform_batch", "Generate", "Batch transform (chained)"),
    ("sqlseed_before_insert", "Write", "Pre-insert callback"),
    ("sqlseed_after_insert", "Write", "Post-insert callback"),
    ("sqlseed_shared_pool_loaded", "Shared Pool", "Shared pool load complete"),
]

print("sqlseed 11 Hook lifecycle:\n")
for i, (name, phase, desc) in enumerate(hookspec_names, 1):
    print(f"  {i:2d}. [{phase}] {name}")
    print(f"      {desc}")

sqlseed 11 Hook lifecycle:

   1. [Register] sqlseed_register_providers
      Register custom Provider
   2. [Register] sqlseed_register_column_mappers
      Register custom column mapping rules
   3. [AI Analysis] sqlseed_ai_analyze_table
      AI analyzes table structure (firstresult)
   4. [AI Analysis] sqlseed_pre_generate_templates
      Pre-generate template values (firstresult)
   5. [Generate] sqlseed_before_generate
      Pre-generation callback
   6. [Generate] sqlseed_after_generate
      Post-generation callback
   7. [Generate] sqlseed_transform_row
      Per-row transform (hot path)
   8. [Generate] sqlseed_transform_batch
      Batch transform (chained)
   9. [Write] sqlseed_before_insert
      Pre-insert callback
  10. [Write] sqlseed_after_insert
      Post-insert callback
  11. [Shared Pool] sqlseed_shared_pool_loaded
      Shared pool load complete


## 4. In Practice: transform_row Hook

Register a plugin that auto-adds a `created_at` timestamp before each row is written.

In [4]:
from datetime import datetime

import pluggy


class TimestampPlugin:
    """Adds created_at timestamp to each row."""

    @hookimpl
    def sqlseed_transform_row(self, table_name: str, row: dict) -> dict | None:
        if 'created_at' not in row or row['created_at'] is None:
            row['created_at'] = datetime.now().isoformat()
            return row
        return None

# Register and use with fill

with connect(str(db_path), provider='mimesis', locale='en') as orch:
    orch._ext.plugins.register(TimestampPlugin())
    result = orch.fill_table('organizations', count=5, clear_before=True, seed=42,
        columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                 "parent_code": {"type": "choice", "choices": [*[r[0] for r in __import__("sqlite3").connect(str(db_path)).execute("SELECT org_code FROM organizations").fetchall()]]}})

print(f'Filled {result.count} rows in {result.elapsed:.3f}s')
conn = sqlite3.connect(str(db_path))
rows = conn.execute('SELECT org_code, name, created_at FROM organizations').fetchall()
for row in rows:
    print(f'  {row[0]}: {row[1]} | created_at={row[2]}')
conn.close()


Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Filled 5 rows in 0.034s
  ORG-0433: Anthony Reilly | created_at=2020-02-01 23:17:15.234053
  ORG-1819: Kai Day | created_at=2004-12-04 21:47:57.571858
  ORG-0013: Cleveland Osborn | created_at=2002-10-14 01:01:05.229258
  ORG-8908: Zack Holder | created_at=2007-09-20 00:35:12.750800
  ORG-8637: Arden Brady | created_at=2020-12-18 13:14:28.617889


## 5. transform_batch Hook

`transform_batch` transforms the entire batch of data, suitable for batch computation or filtering.

In [5]:
class UppercasePlugin:
    """Uppercase all string values in each batch."""

    @hookimpl
    def sqlseed_transform_batch(self, table_name: str, batch: list[dict]) -> list[dict] | None:
        for row in batch:
            for key, val in row.items():
                if isinstance(val, str) and key not in ('created_at', 'registered_at', 'due_at', 'completed_at', 'uploaded_at'):  # noqa: E501
                    row[key] = val.upper()
        return batch

with connect(str(db_path), provider='mimesis', locale='en') as orch:
    orch._ext.plugins.register(UppercasePlugin())
    result = orch.fill_table('organizations', count=3, clear_before=True, seed=42,
        columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                 "parent_code": {"type": "choice", "choices": [*[r[0] for r in __import__("sqlite3").connect(str(db_path)).execute("SELECT org_code FROM organizations").fetchall()]]}})
    print(f'Filled {result.count} rows with UppercasePlugin')

conn = sqlite3.connect(str(db_path))
rows = conn.execute('SELECT org_code, name FROM organizations').fetchall()
for row in rows:
    print(f'  {row[0]}: {row[1]}')
conn.close()


Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Filled 3 rows with UppercasePlugin
  ORG-0433: ANTHONY REILLY
  ORG-1819: KAI DAY
  ORG-0013: CLEVELAND OSBORN


## 6. Package as a Standalone Plugin

Via the `entry-point` mechanism, sqlseed auto-discovers and loads installed plugin packages.

In [6]:
# Minimal pyproject.toml for a sqlseed plugin
print('''[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "sqlseed-my-plugin"
version = "0.1.0"
dependencies = ["sqlseed>=0.1.0"]

[project.entry-points.sqlseed]
my_plugin = "my_plugin.plugin"''')

print('\nPlugin module (my_plugin/plugin.py):')
print('''from sqlseed.plugins.hookspecs import hookimpl

class MyPlugin:
    @hookimpl
    def sqlseed_transform_row(self, table_name, row):
        row["custom_field"] = "processed"
        return row''')

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "sqlseed-my-plugin"
version = "0.1.0"
dependencies = ["sqlseed>=0.1.0"]

[project.entry-points.sqlseed]
my_plugin = "my_plugin.plugin"

Plugin module (my_plugin/plugin.py):
from sqlseed.plugins.hookspecs import hookimpl

class MyPlugin:
    @hookimpl
    def sqlseed_transform_row(self, table_name, row):
        row["custom_field"] = "processed"
        return row


## Summary

| Hook | Phase | Purpose |
|------|------|------|
| `sqlseed_register_providers` | Register | Custom Provider |
| `sqlseed_register_column_mappers` | Register | Custom column mapping |
| `sqlseed_before_generate` | Pre-generate | Preparation |
| `sqlseed_transform_row` | During generate | Per-row transform |
| `sqlseed_transform_batch` | During generate | Batch transform |
| `sqlseed_after_generate` | Post-generate | Cleanup |
| `sqlseed_before_insert` | Pre-insert | Preprocessing |
| `sqlseed_after_insert` | Post-insert | Record stats |

**Next**: [10-cli-reference.ipynb](10-cli-reference.ipynb) — CLI Reference Manual

In [7]:
# ✅ Validation: ensure data was successfully generated and written
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # Basic row count validation
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
